In [22]:
import pandas as pd
import pyarrow as pa

In [20]:
for ext in ['pandas.period', 'pandas.interval']:
    try:
        pa.unregister_extension_type(ext)
    except Exception:
        # We use a broad Exception here so it completely ignores 
        # any PyArrow complaints if the type isn't found
        pass

In [24]:
df_sample_s = pd.read_parquet("parquet/submissions_2024_april_full.parquet", engine="fastparquet")
print(df_sample_s.dtypes) 
display(df_sample_s.head(5).to_dict(orient="list"))

author         object
subreddit      object
count_posts     int64
dtype: object


{'author': ['ckimball6588',
  'NBA_MOD',
  'pop-hon_ula',
  'Emperors_Enterprise',
  'Remarkable_Prompt691'],
 'subreddit': ['funny', 'nba', 'femalefashionadvice', 'canada', 'facepalm'],
 'count_posts': [1, 1, 1, 1, 1]}

In [26]:
df_sample_c = pd.read_parquet("parquet/comments_2024_april_full.parquet", engine="fastparquet")
print(df_sample_c.dtypes) 
display(df_sample_c.head(5).to_dict(orient="list"))

author            object
subreddit         object
count_comments     int64
dtype: object


{'author': ['SapphicLullaby',
  'nimmin13',
  '248kb',
  'Milqy',
  'InfiniteLuxGiven'],
 'subreddit': ['crochet',
  'music',
  'mildlyinfuriating',
  'ufos',
  'unitedkingdom'],
 'count_comments': [1, 1, 1, 1, 1]}

In [27]:
import polars as pl
from pathlib import Path

In [33]:
# AGGREGATION YOU HEAR ME? AGGREGATING !

# analytics/user_subreddit_interactions.parquet 
# with (author, subreddit, interaction_count) 
# aggregated over Jan–Jun yas koriq

# i also generated PER-MONTHs JIC YK... IDK
# but for the next codee, we'r j gna use the agg'd 6-mos 

In [30]:
import polars as pl
from pathlib import Path

PARQUET_DIR = Path("parquet")
OUT_DIR = Path("analytics")
OUT_DIR.mkdir(exist_ok=True)

MONTHS = ["january", "february", "march", "april", "may", "june"]
OVERWRITE = False

def safe_read_parquet(path: Path) -> pl.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"Missing file: {path}")
    return pl.read_parquet(path)

def clean_interactions(df: pl.DataFrame, count_col: str) -> pl.DataFrame:
    return (
        df
        .filter(
            pl.col("author").is_not_null()
            & pl.col("subreddit").is_not_null()
            & (pl.col("author") != "[deleted]")
            & (pl.col("author") != "AutoModerator")
        )
        .with_columns(
            pl.col("subreddit").str.to_lowercase(),
            pl.col(count_col).cast(pl.Int64),
        )
    )

def safe_write_parquet(df: pl.DataFrame, path: Path):
    if path.exists() and not OVERWRITE:
        print(f"[skip] {path} already exists")
        return
    df.write_parquet(path)
    print(f"[write] {path}")

def load_month_interactions(month: str) -> pl.DataFrame:
    sub_path = PARQUET_DIR / f"submissions_2024_{month}_full.parquet"
    com_path = PARQUET_DIR / f"comments_2024_{month}_full.parquet"

    print(f"[load] {month} submissions: {sub_path}")
    df_posts = safe_read_parquet(sub_path)
    print(f"[load] {month} comments  : {com_path}")
    df_comments = safe_read_parquet(com_path)

    df_posts = clean_interactions(df_posts, "count_posts").select(
        pl.col("author"),
        pl.col("subreddit"),
        pl.col("count_posts").alias("interaction_count"),
    )
    df_comments = clean_interactions(df_comments, "count_comments").select(
        pl.col("author"),
        pl.col("subreddit"),
        pl.col("count_comments").alias("interaction_count"),
    )

    df_month = (
        pl.concat([df_posts, df_comments])
        .group_by(["author", "subreddit"])
        .agg(pl.col("interaction_count").sum().cast(pl.Int64))
    )

    return df_month

# streaming-style global aggregation to control memory
df_all = None

for m in MONTHS:
    print(f"[month] {m}")
    df_m = load_month_interactions(m)

    # optional per-month checkpoint
    month_out = OUT_DIR / f"user_subreddit_2024_{m}.parquet"
    safe_write_parquet(df_m, month_out)

    if df_all is None:
        df_all = df_m
    else:
        df_all = (
            pl.concat([df_all, df_m])
            .group_by(["author", "subreddit"])
            .agg(pl.col("interaction_count").sum().cast(pl.Int64))
        )

final_out = OUT_DIR / "user_subreddit_interactions.parquet"
safe_write_parquet(df_all, final_out)

print(df_all.head())
print(df_all.shape)


[month] january
[load] january submissions: parquet\submissions_2024_january_full.parquet
[load] january comments  : parquet\comments_2024_january_full.parquet
[write] analytics\user_subreddit_2024_january.parquet
[month] february
[load] february submissions: parquet\submissions_2024_february_full.parquet
[load] february comments  : parquet\comments_2024_february_full.parquet
[write] analytics\user_subreddit_2024_february.parquet
[month] march
[load] march submissions: parquet\submissions_2024_march_full.parquet
[load] march comments  : parquet\comments_2024_march_full.parquet
[write] analytics\user_subreddit_2024_march.parquet
[month] april
[load] april submissions: parquet\submissions_2024_april_full.parquet
[load] april comments  : parquet\comments_2024_april_full.parquet
[write] analytics\user_subreddit_2024_april.parquet
[month] may
[load] may submissions: parquet\submissions_2024_may_full.parquet
[load] may comments  : parquet\comments_2024_may_full.parquet
[write] analytics\user

In [78]:
# SELECTING IDENTIFYING THEEE TOP 500 SUBREDDITS & ACTIVE USERS

# output for ths are:

## table for top 500 subreddits
### sub_activity: (subreddit, total_interactions) sorted descending

## table for filtered interaction
### df_active: all (author, subreddit, interaction_count) rows
#### where subreddit is an elements of the top 500 ;
#### where author is an active user (in df_active) :
##### user_total_interactions >= 20 (still an elements of the top 500);
##### user_n_subs >= 2 (multi‑community)

# GETS? lord    ok gets 


In [119]:
import polars as pl
from pathlib import Path

ANALYTICS_DIR = Path("analytics")
INTERACTIONS_PATH = ANALYTICS_DIR / "user_subreddit_interactions.parquet"
ACTIVE_PATH = ANALYTICS_DIR / "user_subreddit_active_interactions.parquet"

df_full = pl.read_parquet(INTERACTIONS_PATH)


# lord ito quick sanity check 
print("Full interactions table:", df.shape)

Full interactions table: (46724134, 3)


In [81]:
# Top 500 subreddits by total interactions
sub_activity = (
    df
    .group_by("subreddit")
    .agg(pl.col("interaction_count").sum().alias("total_interactions"))
    .sort("total_interactions", descending=True)
)

# print("sub_activity shape:", sub_activity.shape)
# print("Top 10 subreddits:")
# print(sub_activity.head(10))

# # only taking the 500 most active subreddits
# top500 = sub_activity.head(500)

# # siguraduhin na may saktong 500 unique subreddit names
# top500_subs = top500["subreddit"].unique().to_list()
# print("Unique subreddits in top500 list:", len(top500_subs))
# assert len(top500_subs) == 500, "Top500 must contain 500 unique subreddits"

# # now filter df to only those 500 subreddits
# df_topsubs = df.join(
#     pl.DataFrame({"subreddit": top500_subs}),
#     on="subreddit",
#     how="inner",
# )

# print("Unique subreddits in df:", df["subreddit"].n_unique())
# print("Unique subreddits in df_topsubs:", df_topsubs["subreddit"].n_unique())
# print("After restricting to top 500 subs:", df_topsubs.shape)

print("Total subreddits:", sub_activity.height)  # dapat 500 yan lord
print("Top 10 subreddits:\n", sub_activity.head(10))

Total subreddits: 500
Top 10 subreddits:
 shape: (10, 2)
┌───────────────────┬────────────────────┐
│ subreddit         ┆ total_interactions │
│ ---               ┆ ---                │
│ str               ┆ i64                │
╞═══════════════════╪════════════════════╡
│ askreddit         ┆ 21332107           │
│ aitah             ┆ 7301950            │
│ facepalm          ┆ 5686680            │
│ nostupidquestions ┆ 5379142            │
│ teenagers         ┆ 5204826            │
│ nba               ┆ 5058636            │
│ helldivers        ┆ 4796214            │
│ wallstreetbets    ┆ 4452225            │
│ politics          ┆ 4002790            │
│ nfl               ┆ 4001359            │
└───────────────────┴────────────────────┘


In [121]:
user_stats = (
    df_full
    .group_by("author")
    .agg([
        pl.col("interaction_count").sum().alias("user_total_interactions"),
        pl.col("subreddit").n_unique().alias("user_n_subs"),
    ])
)

In [52]:
# ang median total interactions is 3 lang; median number of subreddits is 1
# mean total interaction is apprx 20; it still include a very large fraction

# new filters: >= 30 interactions in >= 3 subreddits

# active users count = 1,366,020 (about 10% of all 13.36M users)


# reasoning for 30: above normal sha (2x of mean, and far above the 75th %ile),

# Global median interactions = 3; 75th percentile = 9
# with >= 30 interactions and >= 3 subreddits:
 ## active users count = 1,366,020 (about 10% of all 13.36M users)
 ## median interactions among actives = 71 (vs 3 globally)
 ## median subreddits among actives = 12 (vs 1 globally)

In [122]:
ACTIVE_MIN_INTERACTIONS = 30
ACTIVE_MIN_SUBS = 3

active_users = user_stats.filter(
    (pl.col("user_total_interactions") >= ACTIVE_MIN_INTERACTIONS)
    & (pl.col("user_n_subs") >= ACTIVE_MIN_SUBS)
)

active_user_ids = set(active_users["author"].to_list())
print("Active users count:", len(active_user_ids))
print(active_users.select("user_total_interactions").describe())
print(active_users.select("user_n_subs").describe())

df_active = df_full.filter(pl.col("author").is_in(active_user_ids))
print("df_active shape after enforcing filters:", df_active.shape)

Active users count: 1366020
shape: (9, 2)
┌────────────┬─────────────────────────┐
│ statistic  ┆ user_total_interactions │
│ ---        ┆ ---                     │
│ str        ┆ f64                     │
╞════════════╪═════════════════════════╡
│ count      ┆ 1.36602e6               │
│ null_count ┆ 0.0                     │
│ mean       ┆ 143.681262              │
│ std        ┆ 261.762073              │
│ min        ┆ 30.0                    │
│ 25%        ┆ 43.0                    │
│ 50%        ┆ 71.0                    │
│ 75%        ┆ 143.0                   │
│ max        ┆ 39271.0                 │
└────────────┴─────────────────────────┘
shape: (9, 2)
┌────────────┬─────────────┐
│ statistic  ┆ user_n_subs │
│ ---        ┆ ---         │
│ str        ┆ f64         │
╞════════════╪═════════════╡
│ count      ┆ 1.36602e6   │
│ null_count ┆ 0.0         │
│ mean       ┆ 15.417882   │
│ std        ┆ 11.923861   │
│ min        ┆ 3.0         │
│ 25%        ┆ 7.0         │
│ 50%     

In [123]:
ACTIVE_PATH = ANALYTICS_DIR / "user_subreddit_active_interactions.parquet"
df_active.write_parquet(ACTIVE_PATH)
print(f"Saved enforced-active interactions to: {ACTIVE_PATH}")

Saved enforced-active interactions to: analytics\user_subreddit_active_interactions.parquet


In [65]:
# USER FEATURE CONSTRUCTION FOR CLUSTERING


In [88]:
import numpy as np

In [124]:
ANALYTICS_DIR = Path("analytics")
ACTIVE_PATH = ANALYTICS_DIR / "user_subreddit_active_interactions.parquet"
USER_FEATS_PATH = ANALYTICS_DIR / "user_features_for_clustering.parquet"

In [125]:
# load df_active
df_active = pl.read_parquet(ACTIVE_PATH)
print("[3.1] Active interactions table shape:", df_active.shape)

[3.1] Active interactions table shape: (21061135, 3)


In [126]:
# per-user totals and spread
user_totals = (
    df_active
    .group_by("author")
    .agg([
        pl.col("interaction_count").sum().alias("user_total_interactions"),
        pl.col("subreddit").n_unique().alias("user_n_subs"),
    ])
)

print("[3.2] user_totals shape:", user_totals.shape)
print(user_totals.select("user_total_interactions").describe())
print(user_totals.select("user_n_subs").describe())


[3.2] user_totals shape: (1366020, 3)
shape: (9, 2)
┌────────────┬─────────────────────────┐
│ statistic  ┆ user_total_interactions │
│ ---        ┆ ---                     │
│ str        ┆ f64                     │
╞════════════╪═════════════════════════╡
│ count      ┆ 1.36602e6               │
│ null_count ┆ 0.0                     │
│ mean       ┆ 143.681262              │
│ std        ┆ 261.762073              │
│ min        ┆ 30.0                    │
│ 25%        ┆ 43.0                    │
│ 50%        ┆ 71.0                    │
│ 75%        ┆ 143.0                   │
│ max        ┆ 39271.0                 │
└────────────┴─────────────────────────┘
shape: (9, 2)
┌────────────┬─────────────┐
│ statistic  ┆ user_n_subs │
│ ---        ┆ ---         │
│ str        ┆ f64         │
╞════════════╪═════════════╡
│ count      ┆ 1.36602e6   │
│ null_count ┆ 0.0         │
│ mean       ┆ 15.417882   │
│ std        ┆ 11.923861   │
│ min        ┆ 3.0         │
│ 25%        ┆ 7.0         │


In [128]:
# per-user per-subreddit shares
user_sub_shares = (
    df_active
    .group_by(["author", "subreddit"])
    .agg(pl.col("interaction_count").sum().alias("cnt"))
)

user_sub_shares = user_sub_shares.join(
    user_totals.select(["author", "user_total_interactions"]),
    on="author",
    how="left",
)

user_sub_shares = user_sub_shares.with_columns(
    (pl.col("cnt") / pl.col("user_total_interactions")).alias("p")
)

print("[3.3] user_sub_shares shape:", user_sub_shares.shape)
print(user_sub_shares.head())

[3.3] user_sub_shares shape: (21061135, 5)
shape: (5, 5)
┌──────────────────────┬───────────────────┬─────┬─────────────────────────┬──────────┐
│ author               ┆ subreddit         ┆ cnt ┆ user_total_interactions ┆ p        │
│ ---                  ┆ ---               ┆ --- ┆ ---                     ┆ ---      │
│ str                  ┆ str               ┆ i64 ┆ i64                     ┆ f64      │
╞══════════════════════╪═══════════════════╪═════╪═════════════════════════╪══════════╡
│ MajesticRocket       ┆ canada            ┆ 1   ┆ 63                      ┆ 0.015873 │
│ Prestigious_Long5860 ┆ interestingasfuck ┆ 5   ┆ 93                      ┆ 0.053763 │
│ nhbruh               ┆ interestingasfuck ┆ 1   ┆ 62                      ┆ 0.016129 │
│ Habay12              ┆ sipstea           ┆ 4   ┆ 683                     ┆ 0.005857 │
│ eutectic_h8r         ┆ houseofthedragon  ┆ 2   ┆ 1515                    ┆ 0.00132  │
└──────────────────────┴───────────────────┴─────┴─────────────

In [149]:
# aggregate to max_share and entropy per user

def entropy_from_probs(probs):
    arr = np.asarray(probs, dtype=float)
    arr = arr[arr > 0]
    if arr.size == 0:
        return 0.0
    return float(-(arr * np.log(arr)).sum())

# group and collect shares into list per user
user_shares_agg = (
    user_sub_shares
    .group_by("author")
    .agg([
        pl.col("p").max().alias("max_share"),
        pl.col("p").alias("p_list"),   # list[f64] per user
    ])
)

print("[3.4] user_shares_agg shape:", user_shares_agg.shape)
print(user_shares_agg.head())

# move to Python, compute entropy, and rebuild a Polars DF
pdf = user_shares_agg.to_pandas()
pdf["entropy"] = pdf["p_list"].apply(entropy_from_probs)
pdf = pdf.drop(columns=["p_list"])

user_shares_agg = pl.from_pandas(pdf)

print("[3.4b] user_shares_agg with entropy:")
print(user_shares_agg.head())

[3.4] user_shares_agg shape: (1366020, 3)
shape: (5, 3)
┌──────────────────────┬───────────┬─────────────────────────────────┐
│ author               ┆ max_share ┆ p_list                          │
│ ---                  ┆ ---       ┆ ---                             │
│ str                  ┆ f64       ┆ list[f64]                       │
╞══════════════════════╪═══════════╪═════════════════════════════════╡
│ NineteenSixtySix     ┆ 0.097222  ┆ [0.013889, 0.027778, … 0.01388… │
│ StillGalaxy99        ┆ 0.238636  ┆ [0.028409, 0.005682, … 0.00568… │
│ Galileo1632          ┆ 0.25641   ┆ [0.102564, 0.102564, … 0.20512… │
│ 53459803249024083345 ┆ 0.294118  ┆ [0.019608, 0.019608, … 0.07843… │
│ piryo_eobtgo         ┆ 0.296875  ┆ [0.015625, 0.171875, … 0.03125… │
└──────────────────────┴───────────┴─────────────────────────────────┘
[3.4b] user_shares_agg with entropy:
shape: (5, 3)
┌──────────────────────┬───────────┬──────────┐
│ author               ┆ max_share ┆ entropy  │
│ ---           

In [150]:
user_features = (
    user_totals
    .join(user_shares_agg, on="author", how="left")
    .with_columns(
        pl.col("user_total_interactions")
        .cast(pl.Float64)
        .log()
        .alias("log_total_interactions")
    )
)

print("[3.5] user_features shape:", user_features.shape)
print(user_features.head())

[3.5] user_features shape: (1366020, 6)
shape: (5, 6)
┌─────────────────┬──────────────────────┬─────────────┬───────────┬──────────┬──────────────────────┐
│ author          ┆ user_total_interacti ┆ user_n_subs ┆ max_share ┆ entropy  ┆ log_total_interactio │
│ ---             ┆ ons                  ┆ ---         ┆ ---       ┆ ---      ┆ ns                   │
│ str             ┆ ---                  ┆ u32         ┆ f64       ┆ f64      ┆ ---                  │
│                 ┆ i64                  ┆             ┆           ┆          ┆ f64                  │
╞═════════════════╪══════════════════════╪═════════════╪═══════════╪══════════╪══════════════════════╡
│ fleeceflowers77 ┆ 30                   ┆ 6           ┆ 0.733333  ┆ 0.949594 ┆ 3.401197             │
│ eagengabriel    ┆ 69                   ┆ 26          ┆ 0.304348  ┆ 2.646768 ┆ 4.234107             │
│ creepy-cats     ┆ 128                  ┆ 35          ┆ 0.3828125 ┆ 2.617916 ┆ 4.85203              │
│ AshevilleHooker ┆

In [154]:
USER_FEATS_PATH = ANALYTICS_DIR / "user_features_for_clustering.parquet"
user_features.write_parquet(USER_FEATS_PATH)
print(f"[3.6] Saved user features to: {USER_FEATS_PATH}")

[3.6] Saved user features to: analytics\user_features_for_clustering.parquet


In [151]:
# CCLUSTERING ON USER BEHAVIORAL FEATURES

In [152]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

In [153]:
ANALYTICS_DIR = Path("analytics")
USER_FEATS_PATH = ANALYTICS_DIR / "user_features_for_clustering.parquet"
CLUSTERS_PATH = ANALYTICS_DIR / "user_clusters.parquet"

In [157]:
#LOAD features

user_features = pl.read_parquet(USER_FEATS_PATH)
print("[4.1] user_features shape:", user_features.shape)

[4.1] user_features shape: (1366020, 6)


In [158]:
# select feature matrix

feature_cols = ["log_total_interactions", "user_n_subs", "entropy"]
X = user_features.select(feature_cols).to_numpy()

In [159]:
# Standardize

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print("[4.3] X_scaled shape:", X_scaled.shape)


[4.3] X_scaled shape: (1366020, 3)


In [ ]:
# clustering K = {3, 4, 5} avoids underfitting (1-2 clus), and overfragmentation (6+ clus)

# many case studies also focus on small K (often 3-5)

# https://akanshasaxena.com/challenge/ml/30-days-30-ml-projects-day-24/  ----- this use k=5
# https://www.kaggle.com/code/debjeetdas/customer-segmentation-using-kmeans-clustering  ----- k=5